# convert_tract_wacs_to_puma.ipynb

This notebook runs after get_wacs_data runs. It takes the tract-level data in the output of that notebook and aggregates them to get puma-level statistics. 

In [ ]:
import os
import re
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath("../../.."))
import lib.io as lio

In [36]:
year = 2018

In [37]:
df = pd.read_csv(f"wac_tracts_{year}.csv").set_index("w_geocode")
df

,C000,CA01,CA02,CA03,CE01,CE02,CE03,CNS01,CNS02,CNS03,...,CNS15,CNS16,CNS17,CNS18,CNS19,CNS20,CD01,CD02,CD03,CD04
w_geocode,,,,,,,,,,,,,,,,,,,,,
10010201001000,4,1,2,1,1,3,0,0,0,0,...,0,0,0,0,0,0,0,2,1,0
10010201001007,1,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
10010201001016,13,3,8,2,3,5,5,0,0,0,...,0,0,0,13,0,0,0,3,6,1
10010201001018,28,6,15,7,6,9,13,0,0,0,...,0,0,0,0,0,0,2,7,11,2
10010201001022,145,8,92,45,43,31,71,0,0,0,...,145,0,0,0,0,0,8,37,40,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
560459513003087,23,1,13,9,5,5,13,0,17,0,...,0,0,0,0,0,3,1,12,4,5
560459513003101,2,0,2,0,0,0,2,0,2,0,...,0,0,0,0,0,0,0,1,0,1
560459513003104,2,1,1,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [38]:
df.index = df.index.astype("str")
df["tract"] = df.index.str[:-4].str.zfill(11)
df["tract"]

w_geocode
10010201001000     01001020100
10010201001007     01001020100
10010201001016     01001020100
10010201001018     01001020100
10010201001022     01001020100
                      ...     
560459513003087    56045951300
560459513003101    56045951300
560459513003104    56045951300
560459513003106    56045951300
560459513003120    56045951300
Name: tract, Length: 2229999, dtype: object

In [39]:
tract_to_puma = pd.read_csv(
    "../../geometry/equivalencies/tract_to_puma.txt", delimiter=",", dtype=str
)
tract_to_puma.head()

,STATEFP,COUNTYFP,TRACTCE,PUMA5CE
0,01,001,020100,02100
1,01,001,020200,02100
2,01,001,020300,02100
3,01,001,020400,02100
4,01,001,020500,02100


In [40]:
tract_to_puma["tract"] = (
    tract_to_puma["STATEFP"] + tract_to_puma["COUNTYFP"] + tract_to_puma["TRACTCE"]
)
tract_to_puma = tract_to_puma.set_index("tract")
tract_to_puma.head()

,STATEFP,COUNTYFP,TRACTCE,PUMA5CE
tract,,,,
01001020100,01,001,020100,02100
01001020200,01,001,020200,02100
01001020300,01,001,020300,02100
01001020400,01,001,020400,02100
01001020500,01,001,020500,02100


In [41]:
tract_to_puma["PUMA5CE"] = tract_to_puma["STATEFP"] + tract_to_puma["PUMA5CE"]
tract_to_puma["PUMA5CE"] = tract_to_puma["PUMA5CE"].str.zfill(7)
tract_to_puma.head()

,STATEFP,COUNTYFP,TRACTCE,PUMA5CE
tract,,,,
01001020100,01,001,020100,0102100
01001020200,01,001,020200,0102100
01001020300,01,001,020300,0102100
01001020400,01,001,020400,0102100
01001020500,01,001,020500,0102100


In [42]:
df["PUMA"] = tract_to_puma.loc[list(df["tract"])]["PUMA5CE"].values
df

,C000,CA01,CA02,CA03,CE01,CE02,CE03,CNS01,CNS02,CNS03,...,CNS17,CNS18,CNS19,CNS20,CD01,CD02,CD03,CD04,tract,PUMA
w_geocode,,,,,,,,,,,,,,,,,,,,,
10010201001000,4,1,2,1,1,3,0,0,0,0,...,0,0,0,0,0,2,1,0,01001020100,0102100
10010201001007,1,0,1,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,01001020100,0102100
10010201001016,13,3,8,2,3,5,5,0,0,0,...,0,13,0,0,0,3,6,1,01001020100,0102100
10010201001018,28,6,15,7,6,9,13,0,0,0,...,0,0,0,0,2,7,11,2,01001020100,0102100
10010201001022,145,8,92,45,43,31,71,0,0,0,...,0,0,0,0,8,37,40,52,01001020100,0102100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
560459513003087,23,1,13,9,5,5,13,0,17,0,...,0,0,0,3,1,12,4,5,56045951300,5600200
560459513003101,2,0,2,0,0,0,2,0,2,0,...,0,0,0,0,0,1,0,1,56045951300,5600200
560459513003104,2,1,1,0,0,1,1,0,0,0,...,0,0,0,0,0,0,1,0,56045951300,5600200


In [43]:
# sourced from https://lehd.ces.census.gov/doc/help/onthemap/LODESTechDoc.pdf
with open("lodes_dictionary.txt") as f:
    raw = f.read()

# Pattern: <index> <code> <type> <description>
pattern = re.compile(r"^\d+\s+(\S+)\s+(Char\d+|Num)\s+(.*)$")

rows = []
for line in raw.strip().splitlines():
    m = pattern.match(line.strip())
    if m:
        code, dtype, desc = m.groups()
        rows.append((code, dtype, desc))
    else:
        print("NO MATCH:", line)

lodes_dict = pd.DataFrame(rows, columns=["code", "type", "description"]).set_index(
    "code"
)
lodes_dict

,type,description
code,,
w_geocode,Char15,Workplace Census Block Code
C000,Num,Total number of jobs
CA01,Num,Number of jobs for workers age 29 or younger
CA02,Num,Number of jobs for workers age 30 to 54
CA03,Num,Number of jobs for workers age 55 or older
CE01,Num,Number of jobs with earnings $1250/month or less
CE02,Num,Number of jobs with earnings $1251/month to $3...
CE03,Num,Number of jobs with earnings greater than $333...
CNS01,Num,Number of jobs in NAICS sector 11 (Agriculture...


In [44]:
puma_lodes = df.drop(columns=["tract"]).groupby(["PUMA"]).sum()
puma_lodes = puma_lodes.rename(columns=dict(lodes_dict["description"]))
puma_lodes

,Total number of jobs,Number of jobs for workers age 29 or younger,Number of jobs for workers age 30 to 54,Number of jobs for workers age 55 or older,Number of jobs with earnings $1250/month or less,Number of jobs with earnings $1251/month to $3333/month,Number of jobs with earnings greater than $3333/month,"Number of jobs in NAICS sector 11 (Agriculture, Forestry, Fishing and Hunting)","Number of jobs in NAICS sector 21 (Mining, Quarrying, and Oil and Gas Extraction)",Number of jobs in NAICS sector 22 (Utilities),...,Number of jobs in NAICS sector 61 (Educational Services),Number of jobs in NAICS sector 62 (Health Care and Social Assistance),"Number of jobs in NAICS sector 71 (Arts, Entertainment, and Recreation)",Number of jobs in NAICS sector 72 (Accommodation and Food Services),Number of jobs in NAICS sector 81 (Other Services [except Public Administration]),Number of jobs in NAICS sector 92 (Public Administration),Number of jobs for workers with Educational Attainment: Less than high school,"Number of jobs for workers with Educational Attainment: High school or equivalent, no college",Number of jobs for workers with Educational Attainment: Some college or Associate degree,Number of jobs for workers with Educational Attainment: Bachelor's degree or advanced degree
PUMA,,,,,,,,,,,,,,,,,,,,,
0100100,65126,17136,33682,14308,16098,26779,22249,327,187,667,...,5244,8385,547,6459,1295,3035,6271,15596,15924,10199
0100200,55849,12289,30305,13255,11232,16446,28171,589,18,336,...,6750,3403,888,4140,860,3363,4811,12422,13334,12993
0100301,78929,19094,42045,17790,17004,20970,40955,164,74,81,...,4888,4724,1140,8349,1240,747,6405,15300,18471,19659
0100302,73792,18404,39227,16161,19130,26425,28237,13,0,610,...,3284,18803,1187,6280,1788,3981,6420,15549,18625,14794
0100400,38349,9478,20490,8381,8663,17262,12424,339,20,471,...,3249,4185,103,2904,536,1831,4218,10119,9416,5118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5600100,54624,11754,29036,13834,14569,17808,22247,882,1574,370,...,5316,8056,1740,9497,1628,3025,4896,12946,14447,10581
5600200,45443,8889,24613,11941,9694,13269,22480,615,6767,873,...,4983,5484,335,3835,1408,3810,3921,12436,12994,7203
5600300,63021,15408,32819,14794,14940,20399,27682,499,791,227,...,8711,9714,682,6383,1698,7672,5195,13416,16437,12565


In [ ]:
puma_migpuma = lio.load_puma_migpuma(
    "../../geometry/equivalencies/puma_migpuma_2010.csv"
)
puma_migpuma.head()

In [ ]:
migpuma_lodes = puma_lodes.copy()
migpuma_lodes["MIGPUMA"] = puma_migpuma.loc[migpuma_lodes.index, "MIGPUMA"]
migpuma_lodes.head()

In [ ]:
migpuma_lodes = migpuma_lodes.groupby("MIGPUMA").sum()
migpuma_lodes.head()

In [ ]:
for df in [puma_lodes, migpuma_lodes]:
    total_jobs = df["Total number of jobs"]

    df["Proportion of jobs for ages 29 and younger"] = (
        df["Number of jobs for workers age 29 or younger"] / total_jobs
    )
    df["Proportion of jobs for ages 30 to 54"] = (
        df["Number of jobs for workers age 30 to 54"] / total_jobs
    )
    df["Proportion of jobs for ages 55 and older"] = (
        df["Number of jobs for workers age 55 or older"] / total_jobs
    )

    df["Proportion of jobs paying 1250 a month or less"] = (
        df["Number of jobs with earnings $1250/month or less"] / total_jobs
    )
    df["Proportion of jobs paying 1251 to 3333 a month or less"] = (
        df["Number of jobs with earnings $1251/month to $3333/month"] / total_jobs
    )
    df["Proportion of jobs paying more than 3333 a month"] = (
        df["Number of jobs with earnings greater than $3333/month"] / total_jobs
    )
    df["Proportion of agriculture jobs"] = (
        df[
            "Number of jobs in NAICS sector 11 (Agriculture, Forestry, Fishing and Hunting)"
        ]
        / total_jobs
    )
    df["Proportion of extraction jobs"] = (
        df[
            "Number of jobs in NAICS sector 21 (Mining, Quarrying, and Oil and Gas Extraction)"
        ]
        / total_jobs
    )

    df["Proportion of utilities jobs"] = (
        df["Number of jobs in NAICS sector 22 (Utilities)"] / total_jobs
    )

    df["Proportion of construction jobs"] = (
        df["Number of jobs in NAICS sector 23 (Construction)"] / total_jobs
    )

    df["Proportion of manufacturing jobs"] = (
        df["Number of jobs in NAICS sector 31-33 (Manufacturing)"] / total_jobs
    )

    df["Proportion of wholesale trade jobs"] = (
        df["Number of jobs in NAICS sector 42 (Wholesale Trade)"] / total_jobs
    )

    df["Proportion of retail trade jobs"] = (
        df["Number of jobs in NAICS sector 44-45 (Retail Trade)"] / total_jobs
    )

    df["Proportion of transportation jobs"] = (
        df["Number of jobs in NAICS sector 48-49 (Transportation and Warehousing)"]
        / total_jobs
    )

    df["Proportion of information jobs"] = (
        df["Number of jobs in NAICS sector 51 (Information)"] / total_jobs
    )

    df["Proportion of finance jobs"] = (
        df["Number of jobs in NAICS sector 52 (Finance and Insurance)"] / total_jobs
    )

    df["Proportion of real estate jobs"] = (
        df["Number of jobs in NAICS sector 53 (Real Estate and Rental and Leasing)"]
        / total_jobs
    )

    df["Proportion of professional jobs"] = (
        df[
            "Number of jobs in NAICS sector 54 (Professional, Scientific, and Technical Services)"
        ]
        / total_jobs
    )

    df["Proportion of management jobs"] = (
        df[
            "Number of jobs in NAICS sector 55 (Management of Companies and Enterprises)"
        ]
        / total_jobs
    )

    df["Proportion of support/administrative jobs"] = (
        df[
            "Number of jobs in NAICS sector 56 (Administrative and Support and Waste Management and Remediation Services)"
        ]
        / total_jobs
    )

    df["Proportion of educational jobs"] = (
        df["Number of jobs in NAICS sector 61 (Educational Services)"] / total_jobs
    )

    df["Proportion of health jobs"] = (
        df["Number of jobs in NAICS sector 62 (Health Care and Social Assistance)"]
        / total_jobs
    )

    df["Proportion of entertainment jobs"] = (
        df["Number of jobs in NAICS sector 71 (Arts, Entertainment, and Recreation)"]
        / total_jobs
    )

    df["Proportion of hospitality jobs"] = (
        df["Number of jobs in NAICS sector 72 (Accommodation and Food Services)"]
        / total_jobs
    )

    df["Proportion of other services jobs"] = (
        df[
            "Number of jobs in NAICS sector 81 (Other Services [except Public Administration])"
        ]
        / total_jobs
    )

    df["Proportion of public administration jobs"] = (
        df["Number of jobs in NAICS sector 92 (Public Administration)"] / total_jobs
    )

    total_jobs_by_education = (
        df[
            "Number of jobs for workers with Educational Attainment: Less than high school"
        ]
        + df[
            "Number of jobs for workers with Educational Attainment: High school or equivalent, no college"
        ]
        + df[
            "Number of jobs for workers with Educational Attainment: Some college or Associate degree"
        ]
        + df[
            "Number of jobs for workers with Educational Attainment: Bachelor's degree or advanced degree"
        ]
    )

    df["Proportion of jobs with less than high school education"] = (
        df[
            "Number of jobs for workers with Educational Attainment: Less than high school"
        ]
        / total_jobs_by_education
    )

    df["Proportion of jobs with high school education, no college"] = (
        df[
            "Number of jobs for workers with Educational Attainment: High school or equivalent, no college"
        ]
        / total_jobs_by_education
    )

    df["Proportion of jobs with some college or associate degree"] = (
        df[
            "Number of jobs for workers with Educational Attainment: Some college or Associate degree"
        ]
        / total_jobs_by_education
    )

    df["Proportion of jobs with bachelor's degree or advanced degree"] = (
        df[
            "Number of jobs for workers with Educational Attainment: Bachelor's degree or advanced degree"
        ]
        / total_jobs_by_education
    )

In [ ]:
for col in puma_lodes.columns:
    print(col)

In [ ]:
for col in migpuma_lodes.columns:
    print(col)

In [ ]:
puma_lodes.to_csv(f"wac_puma_{year}.csv")
migpuma_lodes.to_csv(f"wac_migpuma_{year}.csv")